# Day 3 · 실험 설계 — 기준 조건 위에서 질문하기
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kehdgus9434/sonic-vibe-coding-lab/blob/main/notebooks/day3_experiment.ipynb)

**오늘의 결과물**: 기준 실험 표 + 나만의 실험 1개(가설·지표·결과·한계) + 결과 영상

| 시간 | 내용 |
|---|---|
| 0:00–0:10 | 환경 준비 · 기준/통제 조건이란 |
| 0:10–0:35 | Mission 3-1 기준 실험 3조건 + 지표 추가 |
| 0:35–1:35 | Mission 3-2 자유 실험 |
| 1:35–1:45 | Mission 3-3 결과 영상 |
| 1:45–2:00 | 검증 체크리스트 · (빨리 끝나면 🚀 심화) |


**AI 사용 규칙 (이 과정은 AI 에이전트 사용을 전제로 합니다)**
- 코드는 어떤 도구로 써도 됩니다: Colab의 Gemini, 웹 채팅(ChatGPT·Claude·Gemini), 웹 코딩 에이전트(Claude Code on the web, Codex) — 설치 없이 브라우저만으로 (`docs/00_setup.md`).
- 대신 **평가는 코드가 아니라 당신의 판단**입니다: 🔮 예측(실행 전)과 🧾 검증 기록(체크 후)은 **본인이 직접** 씁니다.
- `sonic_lab/reference.py`·`check.py`는 정답입니다. 에이전트에게도 읽지 말라고 지시하세요(`AGENTS.md`에 규칙이 있음).

- 오늘은 추가로: 실험 카드의 **가설은 코드를 돌리기 전에 본인이** 씁니다. 해석·한계도 본인 판단으로 씁니다(AI 의견은 참고만).

In [ ]:
#@title 0. 실습 환경 준비 — 처음 한 번 실행 (약 1–2분)
REPO = "https://github.com/kehdgus9434/sonic-vibe-coding-lab"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
# 웹 코딩 에이전트 사용자: PR 댓글의 "Colab에서 내 브랜치로 열기" 링크를 쓰면 이 두 값이 자동으로 채워집니다
import os, sys, subprocess
from pathlib import Path
if Path("/content").exists():                              # Google Colab
    LAB_DIR = Path("/content/sonic-vibe-coding-lab")
    if not LAB_DIR.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO, str(LAB_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(LAB_DIR / "requirements-colab.txt")], check=True)
else:                                                      # 로컬 Jupyter / VS Code (저장소 안에서 실행)
    LAB_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(LAB_DIR)
sys.path.insert(0, str(LAB_DIR))
import numpy as np
import sonic_lab
from sonic_lab import check, reference as ref, render, analysis
lab = sonic_lab.start()

In [ ]:
#@title 🔄 (웹 에이전트 사용자) 에이전트가 Fork에 push한 최신 코드 받기 — 필요할 때만
r = subprocess.run(["git", "-C", str(LAB_DIR), "pull", "--ff-only", "origin", BRANCH], capture_output=True, text=True)
print(r.stdout or r.stderr)
for name in [m for m in sys.modules if m.startswith("my_work")]:
    del sys.modules[name]                                  # 다음 import에서 새 코드를 읽도록
print("완료 — my_work 코드를 쓰는 ✍️ 셀을 다시 실행하세요")

---
## Mission 3-1 · 기준 실험 3조건 (25분)
실제 연구(06→07번 노트북)에서 원인을 가려낸 방법입니다. 같은 샘플·같은 시뮬레이터에서 세 조건을 비교합니다.

| 조건 | 정책 입력 | 의미 |
|---|---|---|
| `reference_pd` | 없음 — 참조 관절각을 PD로 그대로 재생 | 정책 없이도 걸을 수 있나? |
| `0` (G1 모드) | G1 참조 관절각 | 로봇 모션을 추종 |
| `2` (SMPL 모드) | 사람 관절 위치 | 사람 모션을 추종 |

> 🔮 **예측 — 아래 셀을 실행하기 전에 직접 적으세요 (AI에게 맡기지 않기)**
> 세 조건 중 어느 것이 넘어질까요? 모드 0과 모드 2 중 어느 쪽 관절 오차가 더 클까요? 이유는?
>
> 내 예측: ______

In [ ]:
N = 1000   # 20초. 시간이 남으면 None(전체 40초)
baseline = {f"cond:{m}": ref.run_physics(lab.sim, lab.policy, lab.motion, mode=m, max_frames=N) for m in ["reference_pd", 0, 2]}
display(analysis.table(baseline))
analysis.plot(baseline, lab);

> 🧭 아래 설명 + `docs/01_contract.md` + 함수 시그니처를 AI에게 주고 ✍️ 셀을 채운 뒤 ✅ 셀로 확인합니다.

1. **해석**: `reference_pd`가 먼저 넘어지는 이유를 AI와 토론하되, **근거가 되는 수치**(골반 높이 그래프 등)는 직접 찾으세요.
2. **지표 추가**: 관절 오차를 **부위별**(다리 12 / 허리 3 / 팔·손목 14, MuJoCo 순서 계약서 2절)로 나누는 함수를 만들고 모드 0과 2를 비교하세요.

In [ ]:
# ✍️ Mission 3-1 — 부위별 관절 오차 (deg)
def joint_error_by_group(arrays, lab):
    """반환 예: {"legs": 3.1, "waist": 1.2, "arms": 7.8}"""
    raise NotImplementedError

# from my_work.day3 import *

for label in ["cond:0", "cond:2"]:
    print(label, joint_error_by_group(baseline[label][1], lab))

> 🧾 **검증 기록 — 직접 적으세요**
> 예측과 실제가 달랐다면 무엇이 달랐나요? 부위별 오차에서 모드 2가 특히 나쁜 부위가 있다면, 입력 데이터의 어떤 한계와 관련 있을까요?
>
> 근거: ______

---
## Mission 3-2 · 자유 실험 (60분)
아래에서 하나를 고르거나 직접 제안하세요. **실험 카드의 가설을 먼저 채우고** 코드를 작성합니다.

| 카드 | 질문 | 힌트 |
|---|---|---|
| A. 좌우 반전 | 반전 모션도 같은 품질로 추종할까? | `lab.load_motion(mirrored=True)` (공식 `_M` 샘플) |
| B. 외란 강건성 | 골반을 얼마나 세게·오래 밀면 넘어질까? | `run_physics(..., on_frame=fn)`에서 `sim.data.xfrc_applied[sim.pelvis_id, :3]` 설정. **힘을 끄는 것도 잊지 말 것.** 수백 N을 잠깐 미는 정도로는 버팁니다 — 크기와 지속시간을 함께 바꿔 경계를 찾으세요 |
| C. 입력 잡음·지연 | 사람 관절 위치에 잡음 σ cm, 또는 k프레임 지연이 생기면? | `dataclasses.replace(lab.motion, local24=...)` |
| D. 재생 속도 | 0.8배/1.2배 속도 걷기도 따라갈까? | 시간축 재보간: 위치는 선형, 회전은 SLERP (계약서 1절) |
| E. 손목 정보 제거 | 손목 각도를 0으로 바꾸면(근사 데이터의 한계)? | `replace(lab.motion, wrists_il=np.zeros_like(...))` |
| F. 내 데이터 | 다른 동작 데이터도 될까? | 강사가 OpenSim NPZ를 주면 `ref.load_opensim_npz(path)` (근사 변환) |

`run_physics`는 `(summary, arrays)`를 돌려줍니다. 여러 조건을 `{"라벨": 결과}`로 모아 `analysis.table`, `analysis.plot`에 넣으세요.

In [ ]:
# 실험 카드 — "가설"은 코드를 돌리기 전에 본인이 작성
card3 = {
    "가설": "",
    "변인": "",
    "지표": "",
    "통제조건": "",
    "결과": "",
    "해석": "",
    "한계": "",
}

In [ ]:
# ✍️ Mission 3-2 — 실험 코드 (웹 코딩 에이전트 사용자는 my_work/day3.py의 experiment()에 작성)
import dataclasses

def experiment(lab, N=1000):
    """반환: {"라벨": run_physics 결과(summary, arrays), ...} — 첫 항목은 기준 조건"""
    results = {"기준(mode 2)": ref.run_physics(lab.sim, lab.policy, lab.motion, mode=2, max_frames=N)}
    # results["내 조건"] = ...
    return results

# from my_work.day3 import *
results = experiment(lab, N)
display(analysis.table(results))
analysis.plot(results, lab);

In [ ]:
# ✅ 카드 확인 (결과·해석·한계까지 채운 뒤)
check.experiment_card(card3)

---
## Mission 3-3 · 결과 영상 (10분)
실험 결과를 눈으로 확인할 6초 영상을 만듭니다. 영상은 *보여주기*일 뿐, 판단은 표의 수치로 합니다.

In [ ]:
label = list(results)[-1]
video = render.mesh_video(lab, results[label][1], f"outputs/day3_{len(results)}.mp4", label=label, seconds=6)
render.show(video or render.skeleton_video(lab, results[label][1], "outputs/day3_skeleton.mp4", title=label, seconds=6))

---
## 마무리 — 검증 체크리스트 (15분)
`docs/03_checklist.md`로 내 실험 결과를 점검하세요. 채울 수 없는 항목이 다음 연구의 첫 번째 할 일입니다.

---
## 🚀 심화 미션 (정답 없음)

**A. 외란 경계 지도** — 힘(N) × 지속시간(s) 격자에서 넘어짐 여부를 지도로 그리세요. 이진 탐색으로 경계를 효율적으로 찾는 코드를 AI와 만들고, 몇 번의 시뮬레이션으로 찾았는지 기록하세요.

**B. 반전 × 모드** — 좌우 반전 샘플을 모드 0과 모드 2로 각각 돌려 원본과 비교하세요. 정책이 좌우 대칭적으로 학습되었다고 말할 수 있나요? 그 주장의 근거와 한계는?

**C. 재현성** — 같은 조건을 여러 번 돌리면 결과가 완전히 같나요? 스레드 수(`OMP_NUM_THREADS`)나 실행 순서가 바뀌면? 차이가 있다면 그 크기를 결과 해석에 어떻게 반영해야 할까요?

In [ ]:
# 🚀 심화 작업 공간